# Review a path-boundary bug with Daybreak Red

| | |
| --- | --- |
| **Model** | `openai.gpt-5.6-cyber` |
| **Inference provider** | Amazon Bedrock |
| **Problem** | Validate and remediate a path-boundary flaw in a synthetic Python function |
| **Region** | `us-east-2` (Ohio) by default; set `AWS_REGION` to use another approved Region |
| **Discovery endpoint** | `https://bedrock-mantle.us-east-2.api.aws/v1` |
| **Inference endpoint** | `https://bedrock-mantle.us-east-2.api.aws/openai/v1` |
| **Difficulty** | Intermediate |

## What you will take away

By the end, you will have verified separately gated Daybreak Red access through Amazon Bedrock and produced a bounded vulnerability-review package: a verdict, inert reproduction inputs, a defensive patch, focused tests, and residual risks for human validation.

Daybreak Red uses the specialist GPT-5.6 Cyber model. It is intended for separately approved, explicitly authorized work such as controlled vulnerability reproduction, proof-of-concept or exploit validation, penetration testing, red teaming, and mitigation development. This notebook uses Red because it asks for reproducible evidence of a specific security boundary failure as well as a patch and tests.

Daybreak Blue uses GPT-5.6 Sol as the broader defensive starting tier. Blue is a better fit for vulnerability triage, secure code review, detection engineering, incident response, controlled malware analysis, and remediation planning when specialist reproduction capability is not required.

This is a synthetic proof of concept built around one deliberately vulnerable Python function. It does not import the fixture, execute generated code, or contact a target. Do not substitute customer source, production paths, secrets, logs, or real target details.

Daybreak Red has a separate approval path; Blue approval is not enough. See [Trusted Access for Cyber](https://learn.chatgpt.com/docs/cyber-safety), the [Red model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-cyber.html), and the recipe [README](../README.md) before running the notebook.

## Set up the client and read the sample

From `cookbooks/`, run `uv sync --group cybersecurity` and use that environment as the kernel. Set `AWS_REGION=us-east-2`; if you use a named profile, also set `AWS_PROFILE`, then run `aws sts get-caller-identity` and confirm the account and role before opening the notebook.

The AWS token generator derives refreshable short-term Bedrock credentials from the standard AWS credential chain. The generic client below uses `/v1` only for model discovery; `BedrockOpenAI` uses `/openai/v1` for inference. Confirm Red is provisioned for the AWS identity, account, project, model, and Region you intend to use. The fixture is read as text and printed for inspection; it is never imported.

In [ ]:
import os
from pathlib import Path

from aws_bedrock_token_generator import provide_token
from openai import BedrockOpenAI, OpenAI

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or "us-east-2"
)
HOST = f"https://bedrock-mantle.{REGION}.api.aws"
CATALOG_ENDPOINT = f"{HOST}/v1"
INFERENCE_ENDPOINT = f"{HOST}/openai/v1"
MODEL_ID = os.environ.get("MODEL_ID", "openai.gpt-5.6-cyber")
PROBE_MAX_OUTPUT_TOKENS = 128
MAX_OUTPUT_TOKENS = 2400

def bedrock_token() -> str:
    return provide_token(region=REGION)


catalog = OpenAI(
    api_key=bedrock_token(),
    base_url=CATALOG_ENDPOINT,
    max_retries=3,
)
client = BedrockOpenAI(
    aws_region=REGION,
    bedrock_token_provider=bedrock_token,
    max_retries=3,
)

relative_data = Path(
    "06-cybersecurity/02-daybreak-red-vulnerability-validation/data/export_reader.py"
)
candidates = (
    Path.cwd() / "data" / "export_reader.py",
    Path.cwd().parent / "data" / "export_reader.py",
    Path.cwd() / relative_data,
    Path.cwd() / "cookbooks" / relative_data,
)
source_file = next(path for path in candidates if path.exists())
source_text = source_file.read_text()

print(f"Synthetic fixture only · {MODEL_ID} in {REGION}")
print("Discovery endpoint:", CATALOG_ENDPOINT)
print("Inference endpoint:", INFERENCE_ENDPOINT)
print(source_text)

## Discover the model

The Mantle Models API lists models available to this AWS environment. This catches a wrong Region or missing model before any workload request. Discovery is necessary but not sufficient: the next cell still performs a real inference check.

In [ ]:
model_ids = {model.id for model in catalog.models.list().data}
if MODEL_ID not in model_ids:
    raise RuntimeError(
        f"{MODEL_ID} was not returned by {CATALOG_ENDPOINT}/models. "
        "Confirm model approval, AWS identity, and Region."
    )

print(f"Model discovery passed: {MODEL_ID}")

## Check model access

This is a real, small inference request using the same model, Region, endpoint, and credentials as the vulnerability-validation call. Run it before continuing. If Red approval, credentials, IAM permissions, model access, or the Region is wrong, the cell should fail here with the SDK error. The request is billable and uses `store=False`.

In [ ]:
probe = client.responses.create(
    model=MODEL_ID,
    input="Reply with READY to confirm that basic inference is working.",
    max_output_tokens=PROBE_MAX_OUTPUT_TOKENS,
    store=False,
)

probe_text = probe.output_text.strip()
if probe.status != "completed" or not probe_text:
    raise RuntimeError(
        f"Model check did not complete successfully: id={probe.id}, "
        f"status={probe.status!r}"
    )

print(f"Model check passed: {probe_text}")

## Ask for a bounded review

The scope below is deliberately narrow: one source file, one request, and inert path strings against imaginary files. Filesystem access, network access, credentials, persistence, and changes are out of scope.

In [ ]:
instructions = """You are assisting an authorized application-security team.
Work only on the synthetic function supplied in this request. Do not target external
systems, request credentials, add persistence, or propose destructive actions.
Reproduction must use inert path strings and imaginary files. Treat the result as review
material for a human; do not claim that any code or test has been executed."""

task = f"""Authorization and scope
- Target: only the synthetic function below.
- Environment: source text only; imaginary lab root /srv/lab.
- Testing window: this single model request.
- Allowed action: static analysis and inert reproduction inputs.
- Prohibited: filesystem access, network access, credential access, persistence,
  and changes.
- Stop condition: if the source is insufficient, state what is unknown and do not
  propose active validation.

Expected security property
read_export must read only .json or .txt regular files whose resolved path remains
beneath /srv/lab/exports.

Source:
```python
{source_text}
```

Produce a concise validation report with:
1. Verdict and the exact weakness
2. Minimal non-destructive reproduction inputs and why they cross the boundary
3. A corrected implementation that fails closed
4. Focused regression tests, including allowed and denied cases
5. Residual risks and human review checks

Do not execute or claim to execute the examples."""

response = client.responses.create(
    model=MODEL_ID,
    instructions=instructions,
    input=task,
    reasoning={"effort": "low"},
    max_output_tokens=MAX_OUTPUT_TOKENS,
    store=False,
)

output_text = response.output_text.strip()
if response.status != "completed" or not output_text:
    raise RuntimeError(
        f"Vulnerability-validation request did not complete: id={response.id}, "
        f"status={response.status!r}, details={response.incomplete_details!r}"
    )

print(output_text)

## Review and conclude

### What was achieved

- Discovery found the exact model ID, and the access check confirmed that the selected identity can invoke the separately approved Daybreak Red model through the Ohio Bedrock Mantle endpoint.
- The synthetic function was reviewed without being imported or executed, producing a source-linked verdict, inert reproduction strings, a fail-closed patch, regression cases, and residual risks.
- The work stayed inside the written boundary: no filesystem access, target contact, credential access, persistence, or generated-code execution.

### Decide whether the answer is usable

Verify the claimed weakness directly against the source. Check that the reproduction inputs are minimal and non-destructive, the patch handles canonicalization, ancestry, symbolic links, and race conditions, and the tests cover both allowed and denied cases. Portability and operational assumptions should be explicit.

### What this proof of concept does not establish

The patch and tests have not been executed, and no real target has been assessed. The useful outcome is a review package ready for controlled human validation—not authorization to test another system or evidence that the proposed patch is production-ready.